In [1]:
import pandas as pd

stats = pd.read_csv("Data/Player_stats_36min.csv")

In [2]:
seasons = stats.loc[stats["lg"].eq("NBA") & stats["season"].between(1981, 2024)].copy()
seasons["combined"] = seasons["team"].str.fullmatch(r"\d+TM", na=False)
seasons = seasons.sort_values("combined", ascending=False, kind="stable").drop_duplicates(["player_id", "season"])
big_men = seasons.loc[seasons["pos"].isin(["C", "PF", "C-PF", "PF-C", "C/PF", "PF/C"])].drop(columns="combined").copy()

In [3]:
big_men["season_era"] = pd.cut(
    big_men["season"], bins=[1980, 1991, 2002, 2013, 2024],
    labels=["era_1", "era_2", "era_3", "era_4"]
)
era_counts = big_men.groupby(["player_id", "season_era"], observed=True)["season"].nunique().unstack(fill_value=0)
era_counts = era_counts.reindex(columns=["era_1", "era_2", "era_3", "era_4"], fill_value=0)
era = era_counts.idxmax(axis=1).rename("era")
big_men = big_men.join(era, on="player_id")
in_era = big_men.loc[big_men["season_era"].eq(big_men["era"])].copy()

In [4]:
stat_columns = stats.select_dtypes("number").columns.drop("season").tolist()
era_averages = in_era.groupby(["player_id", "era"], as_index=False)[stat_columns].mean()
names = big_men.sort_values("season").drop_duplicates("player_id", keep="last")[["player_id", "player"]]
era_averages = names.merge(era_averages, on="player_id", validate="one_to_one")
era_averages = era_averages[["player_id", "player", "era", *stat_columns]].sort_values(["era", "player"]).reset_index(drop=True)

In [5]:
era_averages.to_csv("Data/big_men_era_averages.csv", index=False)
print(f"{len(era_averages):,} players")
print(era_averages["era"].value_counts().sort_index().to_string())

1,504 players
era
era_1    329
era_2    315
era_3    361
era_4    499
